In [1]:
from datasets import load_dataset

C:\Users\Tarun Sunil\.conda\envs\intern\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = load_dataset("FreedomIntelligence/Medical-R1-Distill-Data", split="train")

In [29]:
print(dataset[0])

{'question': 'A 30-year-old female presents with a history of itching under her right breast and has an annular ring lesion upon examination. What is the most likely fungal organism causing this condition?', 'reasoning (reasoning_content)': "Okay, so I need to figure out the most likely fungal organism causing an annular ring lesion under a 30-year-old female's right breast. Let me start by recalling what annular lesions usually indicate. An annular ring-shaped lesion is typical of certain skin infections. Since the question mentions a fungal organism, it's probably a type of dermatophyte infection.\n\nDermatophytes are fungi that cause infections like tinea corporis (ringworm), tinea cruris (jock itch), and tinea pedis (athlete's foot). The location here is under the breast, which is a warm, moist area. That makes me think of tinea versicolor or maybe candidiasis, but tinea versicolor usually presents with discolored patches, not so much itching. Candidiasis can cause itching and redn

In [5]:
from unsloth import FastLanguageModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Failed to patch SmolVLMForConditionalGeneration forward function.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [8]:
from transformers import AutoTokenizer
from unsloth.chat_templates import get_chat_template
from unsloth.datasets import load_dataset_for_training

ModuleNotFoundError: No module named 'unsloth.datasets'

In [24]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2b-bnb-4bit",  # 4bit quantized version
    max_seq_length = 2048,
    dtype = torch.float16,
    load_in_4bit = True,
)

==((====))==  Unsloth 2025.4.1: Fast Gemma patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 4060 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [25]:
chat_template = get_chat_template(tokenizer)

In [31]:
from transformers import AutoTokenizer

# Load your tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b")

# Define a custom chat template
chat_template = """<|system|>
{{ system_prompt }}
<|user|>
{{ user_input }}
<|assistant|>
"""

# Assign the custom chat template to the tokenizer
tokenizer.chat_template = chat_template


messages = [
    {"role": "system", "content": "You are a helpful medical assistant."},
    {"role": "user", "content": "The patient presents with fever, cough, and fatigue."}
]

# Apply the chat template
formatted_input = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

In [1]:
# from unsloth import FastLanguageModel
from datasets import load_dataset
from transformers import TrainingArguments, Trainer
from trl import SFTTrainer
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

C:\Users\Tarun Sunil\.conda\envs\intern\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import BitsAndBytesConfig

In [3]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    llm_int8_enable_fp32_cpu_offload=True
)

tokenizer = AutoTokenizer.from_pretrained("google/gemma-1.1-2b-it")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-1.1-2b-it",
    quantization_config=bnb_config,
    device_map="cuda:0",
    trust_remote_code=True,
)

Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████| 2/2 [00:06<00:00,  3.39s/it]


In [4]:
model.to(device)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

In [10]:
def format_prompt(example):
    instruction = example["question"]
    input_text = example.get("reasoning (reasoning_content)", "")
    output_text = example["response (content)"]

    prompt = f"### Instruction:\n{instruction.strip()}\n\n"
    if input_text:
        prompt += f"### Input:\n{input_text.strip()}\n\n"
    prompt += f"### Response:\n{output_text.strip()}"

    tokens = tokenizer(prompt, truncation=True, padding="max_length", max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


In [12]:
dataset = load_dataset("FreedomIntelligence/Medical-R1-Distill-Data", split="train")  # Reduce as needed
dataset = dataset.map(format_prompt).map(format_prompt, remove_columns=dataset.column_names)

Map: 100%|███████████████████████████████████████████████████████████████| 22000/22000 [02:49<00:00, 130.16 examples/s]


In [13]:
training_args = TrainingArguments(
    output_dir="gemma-1b-medical-lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    logging_steps=10,
    save_strategy="no",
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit"
)

In [14]:
trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

C:\conda_temp\ipykernel_14532\4088182815.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [16]:
trainer.train()

Step,Training Loss
10,1.595400
20,1.583800
30,1.558800
40,1.541300
50,1.522400
60,1.500700
70,1.569400
80,1.583100
90,1.500700
100,1.439600


TrainOutput(global_step=5500, training_loss=1.4559661353718152, metrics={'train_runtime': 18284.155, 'train_samples_per_second': 1.203, 'train_steps_per_second': 0.301, 'total_flos': 1.3412169420688589e+17, 'train_loss': 1.4559661353718152, 'epoch': 1.0})

In [81]:
torch.cuda.empty_cache()

In [17]:
trainer.save_model("./loc-trained/trained_gemma1-2b")

In [ ]:
model_name = "./loc-trained/trained_gemma1-2b"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,        # Use float16 if you trained in that precision
    device_map="auto"                 # Load model to GPU automatically
)

In [30]:
model.eval()

def generate_response(instruction, input_text=None, max_new_tokens=256, temperature=0.7):
    prompt = f"### Instruction:\n{instruction.strip()}\n\n"
    if input_text:
        prompt += f"### Input:\n{input_text.strip()}\n\n"
    prompt += "### Response:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    return decoded[len(prompt):].strip()

In [31]:
instruction = "Explain what oncogenic viruses are."
input_text = ""
response = generate_response(instruction, input_text)
print("\n", response)


 Okay, so I need to explain what oncogenic viruses are. Let me start by recalling what an oncogene is. An oncogene is a gene that, when activated, leads to uncontrolled cell growth. These genes are often linked to viruses, especially those that cause cancer. 

Now, oncogenic viruses are those that encode oncogenes. They are viruses that infect and replicate in the body of a host cell, producing viral proteins that help the virus replicate and produce new viral particles. These viral proteins can be used to activate the host cell's own cell division machinery to produce new virus particles. 

So, what are some examples of these oncogenic viruses? I remember that the most well-known ones are the human papillomavirus (HPV) and the human immunodeficiency virus (HIV). HPV is responsible for cervical cancer and certain types of skin cancer, while HIV causes acquired immunodeficiency syndrome (AIDS). But there are others too, like the Epstein-Barr virus (EBV), which is linked to nasopharynge